# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Ravikiranbathe/flyrank-ai.git"
REPO_DIR = Path("/content/flyrank-ai")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The key fields show different scales and distributions. Impression, click, and search-volume fields are typically heavy-tailed, so averages alone do not describe every page well. The audit therefore uses summary statistics and grouped comparisons rather than assuming a normal distribution.

In [8]:
# Look at distributions of key search-performance fields.

distribution_cols = [
    "impressions_90d",
    "clicks_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "content_age_days",
]

distribution_cols = [
    c for c in distribution_cols
    if c in df.columns
]

display(df[distribution_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

We test three safe signals using simple grouped comparisons. The goal is to check whether the observed data supports the assumptions behind the signals rather than treating the rules as automatically true.

### Verdicts

- **Signal 1 — Content age vs impressions: MIXED.** The median impressions differ across age groups, but the relationship is not consistently increasing or decreasing across all age tiers.
- **Signal 2 — Position vs CTR: CONFIRMED.** The higher-visibility position groups show higher median CTR than lower-position groups in the observed data.
- **Signal 3 — Trend direction vs impressions: MIXED.** The observed impression levels differ by trend category, but the categories do not form a simple monotonic pattern.

In [9]:
# Test three signals using simple grouped comparisons.

print("Signal test #1: Content age vs impressions")
age_test = (
    df.groupby("age_tier", dropna=False)["impressions_90d"]
    .median()
    .sort_values()
)
display(age_test)

print("\nSignal test #2: Position tier vs CTR")
position_test = (
    df.groupby("position_tier", dropna=False)["ctr"]
    .median()
    .sort_values(ascending=False)
)
display(position_test)

print("\nSignal test #3: Trend direction vs impressions")
trend_test = (
    df.groupby("trend_direction", dropna=False)["impressions_90d"]
    .median()
    .sort_values(ascending=False)
)
display(trend_test)

Signal test #1: Content age vs impressions


,impressions_90d
age_tier,
31-90,294.0
181-365,640.5
91-180,741.5
365+,842.5



Signal test #2: Position tier vs CTR


,ctr
position_tier,
page_1,0.16
striking,0.11
page_3_5,0.03
deep,0.00
top_3,0.00



Signal test #3: Trend direction vs impressions


,impressions_90d
trend_direction,
stable,1944.5
down,961.0
up,587.0
flat,4.0
new,3.0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The baseline uses content age, recent performance trend, CTR, and impressions as practical signals. This test checks whether one of those assumptions is visible in the starter data.

In [10]:
# Flag-linked test: does low CTR correspond to lower click capture
# among pages with meaningful impressions?

flag_test = (
    df.assign(
        ctr_group=np.where(df["ctr"] < 2, "CTR < 2%", "CTR >= 2%")
    )
    .groupby("ctr_group", dropna=False)
    .agg(
        pages=("content_id", "count"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        median_ctr=("ctr", "median")
    )
)

display(flag_test)

print(
    "\nThe table shows the observed difference between the two CTR groups. "
    "This is an association check, not a causal test."
)

,pages,median_impressions,median_clicks,median_ctr
ctr_group,,,,
CTR < 2%,29217,785.0,1.0,0.06
CTR >= 2%,783,22.0,1.0,6.25



The table shows the observed difference between the two CTR groups. This is an association check, not a causal test.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit shows that search-performance signals can differ across content groups, but the differences should be treated as observed associations rather than causal effects. For a content team, these signals are useful for prioritizing pages for review, while individual pages should still be checked by a human before taking action.

In [11]:
# Summarize the practical signals used in the audit.

print("Practical takeaway:")
print("- Use content age, position, CTR, and trend as review signals.")
print("- Use grouped comparisons to check whether the signals hold in the data.")
print("- Treat the results as decision-support, not proof of causation.")

Practical takeaway:
- Use content age, position, CTR, and trend as review signals.
- Use grouped comparisons to check whether the signals hold in the data.
- Treat the results as decision-support, not proof of causation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.